## Load modules

In [3]:
!module load scipy-stack/2023b
!module list


Currently Loaded Modules:
  1) CCconfig              15) code-server/4.101.2
  2) gentoo/2023      (S)  16) calibre/8.6.0
  3) gcccore/.12.3    (H)  17) libreqda/1.0.1
  4) gcc/12.3         (t)  18) flexiblascore/.3.3.1         (H)
  5) hwloc/2.9.1           19) java/17 -> java/17.0.6       (t)
  6) ucx/1.14.1            20) r/4.4.0                      (t)
  7) libfabric/1.18.0      21) rstudio-server/4.4           (t)
  8) pmix/4.2.4            22) python/3.11 -> python/3.11.5 (t)
  9) ucc/1.2.0             23) ipykernel/2023b
 10) openmpi/4.1.5    (m)  24) ipython-kernel/3.11          (S)
 11) flexiblas/3.3.1       25) openrefine/3.9.3
 12) blis/0.9.0            26) jupyterlab-apps/1.0
 13) StdEnv/2023      (S)  27) scipy-stack/2023b            (math)
 14) mii/1.1.2

  Where:
   S:     Module is Sticky, requires --force to unload or purge
   m:     MPI implementations / Implémentations MPI
   math:  Mathematical libraries / Bibliothèques mathématiques
   t:     Tools for developmen

In [5]:
import numpy as np
import pandas as pd
import math as math
import matplotlib.pyplot as plt
import subprocess
import os
import scipy.stats as stats
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import glob

## Remove Dropouts from Data

In [6]:
frequencies_df = pd.read_csv('../1.QC_and_Barcoding/frequencies.csv', sep=',', index_col=0)
frequencies_df

count_col_list = [
    'SC5314_TP0_YPD_1', 'SC5314_TP0_YPD_2', 'SC5314_TP0_YPD_3', 'SC5314_TP0_YPD_4', 
    'SC5314_TP3_YPD_1', 'SC5314_TP3_YPD_2', 'SC5314_TP3_YPD_3', 'SC5314_TP3_YPD_4', 
    'SC5314_TP3_Low_FLZ_1', 'SC5314_TP3_Low_FLZ_2', 'SC5314_TP3_Low_FLZ_3', 'SC5314_TP3_Low_FLZ_4', 
    'SC5314_TP3_High_FLZ_1', 'SC5314_TP3_High_FLZ_2', 'SC5314_TP3_High_FLZ_3', 'SC5314_TP3_High_FLZ_4'
]

freq_col_list = [
    'freq_SC5314_TP0_YPD_1', 'freq_SC5314_TP0_YPD_2', 'freq_SC5314_TP0_YPD_3', 'freq_SC5314_TP0_YPD_4', 
    'freq_SC5314_TP3_YPD_1', 'freq_SC5314_TP3_YPD_2', 'freq_SC5314_TP3_YPD_3', 'freq_SC5314_TP3_YPD_4', 
    'freq_SC5314_TP3_Low_FLZ_1', 'freq_SC5314_TP3_Low_FLZ_2', 'freq_SC5314_TP3_Low_FLZ_3', 'freq_SC5314_TP3_Low_FLZ_4', 
    'freq_SC5314_TP3_High_FLZ_1', 'freq_SC5314_TP3_High_FLZ_2', 'freq_SC5314_TP3_High_FLZ_3', 'freq_SC5314_TP3_High_FLZ_4'
]

The following block basically filters gRNAs that have a frequency that corresponds to them being below the 5th percentile, and also adds a "dynamic epsilon" value to the threshold to ensure that each gRNA has at least a frequency above that which would be present if the count was "1" in a given sample. This is necessary since there are a lot of gRNAs that drop out (and have 0 counts), so if you just filtered with the 5th percentile it wouldn't actually cut anything.

In [7]:
#Define the groups based on your frequency columns
groups = {
    "TP0_YPD": ["freq_SC5314_TP0_YPD_1", "freq_SC5314_TP0_YPD_2", "freq_SC5314_TP0_YPD_3", "freq_SC5314_TP0_YPD_4"],
    "TP3_YPD": ["freq_SC5314_TP3_YPD_1", "freq_SC5314_TP3_YPD_2", "freq_SC5314_TP3_YPD_3", "freq_SC5314_TP3_YPD_4"],
    "TP3_Low_FLZ": ["freq_SC5314_TP3_Low_FLZ_1", "freq_SC5314_TP3_Low_FLZ_2", "freq_SC5314_TP3_Low_FLZ_3", "freq_SC5314_TP3_Low_FLZ_4"],
    "TP3_High_FLZ": ["freq_SC5314_TP3_High_FLZ_1", "freq_SC5314_TP3_High_FLZ_2", "freq_SC5314_TP3_High_FLZ_3", "freq_SC5314_TP3_High_FLZ_4"]
}

#We have 4 replicates for each sample here, so we can make it so that the sgRNA only needs to pass the threshold in 3 replicates in every group (this is still usable)
min_replicates = 3
valid_gRNAs = set(frequencies_df.index)
threshold_values = {}

#Compute total counts per sample (sum of original counts)
total_counts = frequencies_df[count_col_list].sum(axis=0)

for group, samples in groups.items():
    group_df = frequencies_df[samples].copy()
    
    #Compute per-sample thresholds: 5th percentile + epsilon
    thresholds = {}
    for sample in samples:
        #5th percentile of non-zero frequencies
        percentile_5 = np.percentile(group_df[sample][group_df[sample] > 0], 5) if any(group_df[sample] > 0) else 0
        #Dynamic epsilon: frequency that a single count contributes
        epsilon = 1 / total_counts[sample.replace("freq_", "")]
        thresholds[sample] = percentile_5 + epsilon
        threshold_values[sample] = thresholds[sample]
    
    #Compare each row to the per-sample thresholds
    valid_gRNAs_group = group_df.ge(pd.Series(thresholds)).sum(axis=1) >= min_replicates
    valid_gRNAs &= set(valid_gRNAs_group[valid_gRNAs_group].index)

#Metadata columns to keep
metadata_cols = ["Alias", "Gene", "Barcode", "Position"]

#Combine metadata, counts, and frequency columns for filtered gRNAs
filtered_df = frequencies_df.loc[list(valid_gRNAs), metadata_cols + count_col_list + freq_col_list]

#Save to CSV
filtered_df.to_csv("filtered_gRNAs.csv", index=False)

#Optional: print the threshold
common_threshold = min(threshold_values.values())
print("Common Frequency Threshold:", common_threshold)

Common Frequency Threshold: 8.921604556517076e-08


In [8]:
filtered_df

,Alias,Gene,Barcode,Position,SC5314_TP0_YPD_1,SC5314_TP0_YPD_2,SC5314_TP0_YPD_3,SC5314_TP0_YPD_4,SC5314_TP3_YPD_1,SC5314_TP3_YPD_2,...,freq_SC5314_TP3_YPD_3,freq_SC5314_TP3_YPD_4,freq_SC5314_TP3_Low_FLZ_1,freq_SC5314_TP3_Low_FLZ_2,freq_SC5314_TP3_Low_FLZ_3,freq_SC5314_TP3_Low_FLZ_4,freq_SC5314_TP3_High_FLZ_1,freq_SC5314_TP3_High_FLZ_2,freq_SC5314_TP3_High_FLZ_3,freq_SC5314_TP3_High_FLZ_4
Number,,,,,,,,,,,,,,,,,,,,,
1,CR07390CA_242,CR07390C_A,TGCGCACAATTTCCTGCACA,1605752.0,16777.0,25374.0,37679.0,24299.0,53161.0,47017.0,...,0.002546,0.002636,0.001147,1.030577e-03,0.001171,0.000965,0.001388,0.001610,1.423942e-03,0.001372
2,CR07390CA_276_revcom,CR07390C_A,GAGTATAGTGATCCATGTGC,1605740.0,1630.0,1138.0,1558.0,1063.0,212.0,155.0,...,0.000013,0.000010,0.000008,8.273739e-08,0.000014,0.000004,0.000006,0.000005,9.301341e-08,0.000002
3,CR07390CA_239_revcom,CR07390C_A,GCTATAACGTTACTAGTAGT,1605777.0,2233.0,2024.0,2230.0,1768.0,81.0,597.0,...,0.000048,0.000012,0.000224,2.487913e-04,0.000235,0.000235,0.000182,0.000032,8.957191e-05,0.000167
6,CR07390CA_70,CR07390C_A,CTCTTTTGCTCTCATTTTTA,1605924.0,3537.0,2787.0,4190.0,2699.0,3875.0,3517.0,...,0.000229,0.000224,0.000141,1.763134e-04,0.000239,0.000094,0.000135,0.000116,2.015601e-04,0.000138
7,CR10440WA_195_revcom,CR10440W_A,AGGGGTTTAAGCATACTATC,2222654.0,2499.0,1995.0,2960.0,2248.0,2791.0,2333.0,...,0.000083,0.000132,0.000027,8.389571e-05,0.000058,0.000053,0.000089,0.000051,3.674030e-05,0.000080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5692,non-targeting52,non-targeting52,ACGATTTTGACCGAGCCCCA,NaN,5104.0,5851.0,7459.0,5822.0,8716.0,6515.0,...,0.000429,0.000403,0.000071,9.465157e-05,0.000158,0.000104,0.000160,0.000125,7.524785e-05,0.000112
5695,non-targeting55,non-targeting55,AGTGTCCCGTCATGCTCCAG,NaN,7059.0,5164.0,5634.0,4665.0,1735.0,1252.0,...,0.000126,0.000115,0.000047,4.318892e-05,0.000101,0.000006,0.000031,0.000014,2.232322e-05,0.000006
5698,non-targeting58,non-targeting58,GACCTCTTTCGACTAGGCCA,NaN,6683.0,7238.0,9541.0,7239.0,12218.0,11650.0,...,0.000507,0.000530,0.000430,2.255421e-04,0.000261,0.000252,0.000272,0.000250,4.450692e-04,0.000266


## Calculate Log2 Fold-Changes w/ Statistics

In [12]:
#Make a copy so we don't touch the original
statistics_df = filtered_df.copy()

def benjamini_hochberg(p_values):
    
    #Compute BH FDR-adjusted p-values.
    p_values = np.array(p_values)
    n = len(p_values)
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    adjusted = np.empty(n, dtype=float)
    cummin = 1.0
    for i in reversed(range(n)):
        rank = i + 1
        fdr = sorted_p[i] * n / rank
        cummin = min(cummin, fdr)
        adjusted[sorted_indices[i]] = cummin
    return np.minimum(adjusted, 1.0)

#Initialize lists
log2_fc_ypd, log2_fc_low_flz, log2_fc_high_flz = [], [], []
p_ypd, p_low_flz, p_high_flz = [], [], []

for gRNA in statistics_df.index:
    
    #Extract replicate frequencies
    tp0 = statistics_df.loc[gRNA, ['freq_SC5314_TP0_YPD_1','freq_SC5314_TP0_YPD_2','freq_SC5314_TP0_YPD_3','freq_SC5314_TP0_YPD_4']]
    tp3_ypd = statistics_df.loc[gRNA, ['freq_SC5314_TP3_YPD_1','freq_SC5314_TP3_YPD_2','freq_SC5314_TP3_YPD_3','freq_SC5314_TP3_YPD_4']]
    tp3_low_flz = statistics_df.loc[gRNA, ['freq_SC5314_TP3_Low_FLZ_1','freq_SC5314_TP3_Low_FLZ_2','freq_SC5314_TP3_Low_FLZ_3','freq_SC5314_TP3_Low_FLZ_4']]
    tp3_high_flz = statistics_df.loc[gRNA, ['freq_SC5314_TP3_High_FLZ_1','freq_SC5314_TP3_High_FLZ_2','freq_SC5314_TP3_High_FLZ_3','freq_SC5314_TP3_High_FLZ_4']]

    #Filter replicates below threshold and convert to float (ie. even if a sgRNA passed from earlier, we don't want to include a replicate of that sgRNA in our calculations that did not meet the threhold)
    tp0_filtered = tp0[tp0 >= [threshold_values[s] for s in tp0.index]].dropna().astype(float)
    tp3_ypd_filtered = tp3_ypd[tp3_ypd >= [threshold_values[s] for s in tp3_ypd.index]].dropna().astype(float)
    tp3_low_flz_filtered = tp3_low_flz[tp3_low_flz >= [threshold_values[s] for s in tp3_low_flz.index]].dropna().astype(float)
    tp3_high_flz_filtered = tp3_high_flz[tp3_high_flz >= [threshold_values[s] for s in tp3_high_flz.index]].dropna().astype(float)

    #Skip gRNAs with too few replicates
    if len(tp0_filtered) < 3 or len(tp3_ypd_filtered) < 3 or len(tp3_low_flz_filtered) < 3 or len(tp3_high_flz_filtered) < 3:
        log2_fc_ypd.append(np.nan)
        log2_fc_low_flz.append(np.nan)
        log2_fc_high_flz.append(np.nan)
        p_ypd.append(np.nan)
        p_low_flz.append(np.nan)
        p_high_flz.append(np.nan)
        continue

    #Compute log2 fold changes
    tp0_mean = np.mean(tp0_filtered)
    log2_fc_ypd.append(np.log2(np.mean(tp3_ypd_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)
    log2_fc_low_flz.append(np.log2(np.mean(tp3_low_flz_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)
    log2_fc_high_flz.append(np.log2(np.mean(tp3_high_flz_filtered)/tp0_mean) if tp0_mean > 0 else np.nan)

    #Compute p-values
    p_ypd.append(stats.ttest_ind(tp3_ypd_filtered, tp0_filtered, nan_policy='omit')[1])
    p_low_flz.append(stats.ttest_ind(tp3_low_flz_filtered, tp0_filtered, nan_policy='omit')[1])
    p_high_flz.append(stats.ttest_ind(tp3_high_flz_filtered, tp0_filtered, nan_policy='omit')[1])

#Adjust p-values using BH FDR
adj_p_ypd = benjamini_hochberg(p_ypd)
adj_p_low_flz = benjamini_hochberg(p_low_flz)
adj_p_high_flz = benjamini_hochberg(p_high_flz)

#Add results to statistics_df with original column names for log2 fold changes
statistics_df['log2_fold_change_ypd'] = log2_fc_ypd
statistics_df['log2_fold_change_low_flz'] = log2_fc_low_flz
statistics_df['log2_fold_change_high_flz'] = log2_fc_high_flz
statistics_df['p_value_ypd'] = p_ypd
statistics_df['p_value_low_flz'] = p_low_flz
statistics_df['p_value_high_flz'] = p_high_flz
statistics_df['adj_p_value_ypd'] = adj_p_ypd
statistics_df['adj_p_value_low_flz'] = adj_p_low_flz
statistics_df['adj_p_value_high_flz'] = adj_p_high_flz

#Save to CSV
statistics_df.to_csv("statistics_df_with_log2fc_and_fdr.csv")